# `EukaryoticToeholdGate` — usage example (trailing-Kozak layout)

A real, end-to-end run of the single-input eukaryotic toehold switch —
`EukaryoticToeholdGate` in `engine.gates.toehold`. This class is fully implemented
and tested (`tests/engine/gates/test_toehold.py`, 45 passing tests as of this
writing). This is a sibling to [`toehold.ipynb`](toehold.ipynb) (which drives the
gate through a stub `FoldEngine` for fast, dependency-free iteration and covers both
hosts generically) — here we build `EukaryoticToeholdGate` specifically and use the
**real** `FoldEngine` (ViennaRNA) throughout, so every number below is a genuine
fold, not a placeholder.

`ToeholdGate` builds two structurally different eukaryotic layouts (commits
`d754812`, `ee2f5f6`):

* **`"loop"`** — Kozak embedded in the hairpin loop, ported unmodified from the
  prokaryotic mechanism (steric occlusion of the start codon). Unvalidated for a
  eukaryotic toehold specifically — see `KOZAK_LAYOUTS`'s docstring. It also turns
  out Kozak and AUG are *not* adjacent in this layout (a 6 nt stem-closing segment
  sits between them), which breaks the Kozak consensus's own adjacency requirement.
* **`"trailing"`** — Kozak and the start codon sit *after* the closed hairpin
  instead, modelling scanning-ribosome blockage (docs/modalities.md) rather than
  direct start-codon occlusion. This is the layout the team's own eukaryotic
  scripts actually build (`plasmid_prefix + trg_bind_region + loop + stem_down +
  kozak` — Kozak last), and Kozak is genuinely adjacent to AUG here. It also carries
  no trailing `LINKER_SEQUENCE` — cap-dependent scanning initiates the instant the
  40S subunit meets Kozak+AUG, so nothing after the start codon matters to finding
  it, and the payload attaches directly.

It also optionally takes the real effector gene (`payload`, commit `74bf7b4`) and
folds its own first nucleotides into every design instead of a placeholder — because
the real downstream sequence can change which design actually scores best, not just
which one looks best in isolation.

This notebook builds the gate restricted to **`"trailing"`** only
(`kozak_layouts=("trailing",)`), pools designs across the **top 150 trigger
candidates** (not just the single best-scoring one), and reserves at least 40 of
those 150 from the 3&#8242; UTR (from 100 nt before the CDS ends to the end of the
transcript) so that region gets a real chance to compete rather than being crowded
out by however TriggerScorer's global ranking happens to fall.

No Django, no worker, no pipeline — just the gate class, constructed and called
directly, the way `pipeline.py` would use it internally.

## Setup

In [ ]:
# Put <repo>/src on the path. Search upward from cwd for pyproject.toml so this works
# wherever Jupyter is launched from.
import sys
from pathlib import Path

for _base in (Path.cwd(), *Path.cwd().parents):
    if (_base / "pyproject.toml").exists():
        _src = _base / "src"
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break

from engine.domain import Host, Regulation, SelectedGene, TriggerSet, Constraints
from engine.gates.toehold import EukaryoticToeholdGate
from engine.gates.tools.folding import FoldEngine
from engine.gates.tools.translation import TranslationScorer
from engine.gates.tools.codons import CodonOptimizer
from engine.stages.folding import FoldProfiler
from engine.stages.motifs import MotifScreener
from engine.stages.off_target import OffTargetScanner
from engine.stages.triggers import TriggerScorer
from engine import sequences as sq

## 1. Build the tools, once

Per `CLAUDE.md` §5: tools are constructed once and handed to the gate, never built
inside a stage or family. `FoldEngine`'s cache is only useful if every caller shares
one instance — a second `FoldEngine()` means a cold cache and, worse, a second
chance to fold at a different temperature.

In [ ]:
host = Host.HUMAN  # HUMAN | YEAST both take the eukaryotic (Kozak) track

folder = FoldEngine(temperature=37.0)
translation = TranslationScorer(host)
codons = CodonOptimizer(host)

# The real effector gene — from `eff` in CERNAL_FUNCTIONS.py (a GFP-family CDS, already
# RNA, 759 nt, tandem stop UAA UAA). When set, the gate folds its real first
# PAYLOAD_HEAD_LENGTH nt into every design instead of a placeholder — see
# PAYLOAD_HEAD_LENGTH's docstring (commit 74bf7b4) for why this can change which
# design scores best. Set to None to see the placeholder behaviour instead.
PAYLOAD_CDS = (
    "AUGCGUAAAGGAGAAGAACUUUUCACUGGAGUUGUCCCAAUUCUUGUUGAAUUAGAUGGUGAUGUUAAUGGGCACAAAUUUUCUGUCAG"
    "UGGAGAGGGUGAAGGUGAUGCAACAUACGGAAAACUUACCCUUAAAUUUAUUUGCACUACUGGAAAACUACCUGUUCCGUGGCCAACAC"
    "UUGUCACUACUUUCGGUUAUGGUGUUCAAUGCUUUGCGAGAUACCCAGAUCACAUGAAACAGCAUGACUUUUUCAAGAGUGCCAUGCCC"
    "GAAGGUUACGUACAGGAAAGAACUAUAUUUUUCAAAGAUGACGGGAACUACAAGACACGUGCUGAAGUCAAGUUUGAAGGUGAUACCCU"
    "UGUUAAUAGAAUCGAGUUAAAAGGUAUUGAUUUUAAAGAAGAUGGAAACAUUCUUGGACACAAAUUGGAAUACAACUAUAACUCACACA"
    "AUGUAUACAUCAUGGCAGACAAACAAAAGAAUGGAAUCAAAGUUAACUUCAAAAUUAGACACAACAUUGAAGAUGGAAGCGUUCAACUA"
    "GCAGACCAUUAUCAACAAAAUACUCCGAUUGGCGAUGGCCCUGUCCUUUUACCAGACAACCAUUACCUGUCCACACAAUCUGCCCUUUC"
    "GAAAGAUCCCAACGAAAAGAGAGACCACAUGGUCCUUCUUGAGUUUGUAACCGCUGCUGGGAUUACACAUGGCAUGGAUGAACUAUACA"
    "AAAGGCCUGCAGCAAACGACGAAAACUACGCUGCAUCAGUUUAAUAA"
)

# kozak_layouts restricts generate_designs to the "trailing" layout only — the default
# (omit this argument) sweeps both "loop" and "trailing" and lets engine.scoring rank
# across them. This mirrors how `host` is a constructor parameter rather than a
# subclass (docs/engine.md §2.4).
gate = EukaryoticToeholdGate(
    host, folder, translation, codons, kozak_layouts=("trailing",), payload=PAYLOAD_CDS
)
print(gate.required_tools())
print("kozak_layouts:", gate.kozak_layouts)
print("payload_head:", gate.payload_head)

## 2. Pick trigger candidates — via the real `TriggerScorer` (stage 2)

`TriggerScorer.score` is fully implemented (`tests/engine/test_triggers.py`, 47
passing tests): it slides every window of every length in
`constraints.trigger_lengths` across the transcript, screens out forbidden motifs,
folds each survivor for `openness`/`accessibility`/`mfe` via the same `FoldEngine`
the gate uses, checks off-targets, ranks by `accessibility * segment_specificity`,
and yields the top candidates per gene — so we use it for real here instead of
hand-picking one window.

`OffTargetScanner` itself is still a stub (`find_similar`/`scan_trigger` raise
`NotImplementedError`) — **except** when handed an empty transcriptome, which is a
deliberate early-return for exactly this case (a `direct` submission, or a demo like
this one, with no reference index to scan against): `scan_trigger` returns a clean
`OffTargetReport(hits=(), penalty=0.0)` rather than raising. That is a real,
documented behaviour of the class, not a workaround.

Earlier versions of this notebook carried only `candidates[0]`, then the top 5,
through — which never gave a weaker-looking trigger the chance to turn out to fold
into a better switch. This version carries the top 150 through instead, with a floor
on how many must come from the 3&#8242; UTR rather than leaving that to chance — see the
cell below.

In [ ]:
# The full-length human AREG mRNA, as cDNA/DNA notation (T, not U) — same alphabet trap
# CLAUDE.md warns about, so normalise with to_rna() before anything else touches it.
transcript_dna = (
    "AGACGTTCGCACACCTGGGTGCCAGCGCCCCAGAGGTCCCGGGACAGCCCGAGGCGCCGCGCCCGCCGCCCCGAGCTCCCC"
    "AAGCCTTCGAGAGCGGCGCACACTCCCGGTCTCCACTCGCTCTTCCAACACCCGCTCGTTTTGGCGGCAGCTCGTGTCCCA"
    "GAGACCGAGTTGCCCCAGAGACCGAGACGCCGCCGCTGCGAAGGACCAATGAGAGCCCCGCTGCTACCGCCGGCGCCGGTG"
    "GTGCTGTCGCTCTTGATACTCGGCTCAGGCCATTATGCTGCTGGATTGGACCTCAATGACACCTACTCTGGGAAGCGTGAA"
    "CCATTTTCTGGGGACCACAGTGCTGATGGATTTGAGGTTACCTCAAGAAGTGAGATGTCTTCAGGGAGTGAGATTTCCCCT"
    "GTGAGTGAAATGCCTTCTAGTAGTGAACCGTCCTCGGGAGCCGACTATGACTACTCAGAAGAGTATGATAACGAACCACAA"
    "ATACCTGGCTATATTGTCGATGATTCAGTCAGAGTTGAACAGGTAGTTAAGCCCCCCCAAAACAAGACGGAAAGTGAAAAT"
    "ACTTCAGATAAACCCAAAAGAAAGAAAAAGGGAGGCAAAAATGGAAAAAATAGAAGAAACAGAAAGAAGAAAAATCCATGT"
    "AATGCAGAATTTCAAAATTTCTGCATTCACGGAGAATGCAAATATATAGAGCACCTGGAAGCAGTAACATGCAAATGTCA"
    "GCAAGAATATTTCGGTGAACGGTGTGGGGAAAAGTCCATGAAAACTCACAGCATGATTGACAGTAGTTTATCAAAAATTG"
    "CATTAGCAGCCATAGCTGCCTTTATGTCTGCTGTGATCCTCACAGCTGTTGCTGTTATTACAGTCCAGCTTAGAAGACAA"
    "TACGTCAGGAAATATGAAGGAGAAGCTGAGGAACGAAAGAAACTTCGACAAGAGAATGGAAATGTACATGCTATAGCATA"
    "ACTGAAGATAAAATTACAGGATATCACATTGGAGTCACTGCCAAGTCATAGCCATAAATGATGAGTCGGTCCTCTTTCCA"
    "GTGGATCATAAGACAATGGACCCTTTTTGTTATGATGGTTTTAAACTTTCAATTGTCACTTTTTATGCTATTTCTGTATA"
    "TAAAGGTGCACGAAGGTAAAAAGTATTTTTTCAAGTTGTAAATAATTTATTTAATATTTAATGGAAGTGTATTTATTTTA"
    "CAGCTCATTAAACTTTTTTAACCAAA"
)
transcript = sq.to_rna(transcript_dna)
assert sq.is_valid_rna(transcript)
print(f"transcript length: {len(transcript)} nt")

## Locate the CDS, to define the 3&#8242; UTR

Nothing in this engine parses gene annotation (no GTF/GFF loader exists — the CSV/
sequence-database parser is one of the two things `CLAUDE.md` §7 names as having "no
home yet"). To split this transcript into CDS and 3&#8242; UTR at all, approximate the
CDS as the **longest open reading frame** — every AUG in every frame, paired with its
nearest downstream in-frame stop, keeping the longest. This is a heuristic standing in
for real annotation, not something to trust for a transcript whose true CDS isn't
already known by other means — flagged, not silently treated as ground truth.

In [ ]:
def longest_orf(sequence):
    """(start, stop_codon_start, length) of the longest AUG-to-in-frame-stop ORF,
    across all three frames. A heuristic CDS finder, not a gene-model parser."""
    best = None
    for frame in (0, 1, 2):
        augs = sq.find_augs(sequence, frame=frame)
        stops = sq.find_stops(sequence, frame=frame)
        for start in augs:
            downstream_stops = [s for s in stops if s > start]
            if not downstream_stops:
                continue
            stop = min(downstream_stops)
            length = stop - start
            if best is None or length > best[2]:
                best = (start, stop, length, frame)
    return best


orf_start, stop_codon_start, orf_length, orf_frame = longest_orf(transcript)
cds_end = stop_codon_start + 3  # exclusive, i.e. transcript[cds_end:] is the 3' UTR

print(f"longest ORF: {orf_start}-{stop_codon_start} ({orf_length} nt, frame {orf_frame})")
print(f"cds_end: {cds_end}  (transcript length {len(transcript)}, "
      f"3' UTR is {len(transcript) - cds_end} nt)")

In [ ]:
# Stage-2 tools, built once (same injection rule as the gate's own tools).
profiler = FoldProfiler()
screener = MotifScreener()
off_target = OffTargetScanner(transcriptome={})  # empty: no reference index for this demo
scorer = TriggerScorer(profiler, off_target, screener, folder)  # shares the gate's FoldEngine

# TriggerScorer.TOP_K_PER_GENE defaults to 50 — a search-budget knob (its own
# docstring), not a ceiling meaningful here. Raised so the 3' UTR quota below has the
# whole ranked pool to draw from, not just whatever survived a 50-candidate cutoff.
scorer.TOP_K_PER_GENE = 5000

# In a real run this comes from GeneSelector (stage 1); stand in with a minimal
# SelectedGene since this demo starts from a single known transcript.
gene = SelectedGene(
    gene_id="AREG",
    symbol="AREG",
    regulation=Regulation.UP,
    log2_fold_change=2.0,
    score=1.0,
)
constraints = Constraints(trigger_lengths=(30, 36), max_switch_length=200)

candidates = list(scorer.score([gene], {"AREG": transcript}, constraints))
print(f"{len(candidates)} candidate(s) survived screening, best-scoring first\n")
for c in candidates[:5]:
    print(
        f"  {c.trigger_id:22} start={c.start_index:4} len={c.length:2}  "
        f"score={c.score:.3f}  accessibility={c.accessibility:.3f}  gc={c.gc_content:.1f}"
    )

# Select the top N_TRIGGERS, with a floor on how many come from the 3' UTR — a window
# starting anywhere from 100 nt before the CDS ends onward. Nothing in TriggerScorer
# enforces a region quota (it ranks globally by score alone), so this is done here,
# not in the engine: reserve MIN_FROM_UTR slots from the UTR-only pool, then fill the
# rest with whatever scores best overall (UTR leftovers included, so a strong UTR
# candidate isn't capped out of the general competition).
N_TRIGGERS = 150
MIN_FROM_UTR = 40
utr_start = cds_end - 100

utr_pool = [c for c in candidates if c.start_index >= utr_start]
orf_pool = [c for c in candidates if c.start_index < utr_start]
print(f"\n{len(utr_pool)} candidate(s) start in the 3' UTR (>= {utr_start}), "
      f"{len(orf_pool)} start in or before the CDS")

utr_reserved = utr_pool[:MIN_FROM_UTR]
remaining = sorted(utr_pool[MIN_FROM_UTR:] + orf_pool, key=lambda c: c.score, reverse=True)
top_triggers = sorted(
    utr_reserved + remaining[: N_TRIGGERS - len(utr_reserved)],
    key=lambda c: c.score,
    reverse=True,
)

n_utr_selected = sum(1 for c in top_triggers if c.start_index >= utr_start)
print(f"\nselected {len(top_triggers)} trigger(s): {n_utr_selected} from the 3' UTR "
      f"(floor was {MIN_FROM_UTR}), {len(top_triggers) - n_utr_selected} from the CDS")

## 3. Wrap each trigger in its own `TriggerSet`

`TriggerSet` is the circuit's inputs (one activator each here — every trigger builds
its own single-input switch, independently). `Constraints` were already built above,
since `TriggerScorer` needed them too — a run builds `Constraints` once from
`params["constraints"]` and threads the same object through every stage.

In [ ]:
trigger_sets = [TriggerSet(activators=(t,)) for t in top_triggers]

print(f"{len(trigger_sets)} TriggerSet(s) built, e.g.:")
for ts in trigger_sets[:3]:
    print(" ", ts.activators[0].trigger_id, "| arity:", ts.arity, "| logic:", ts.logic_type)

## 4. `is_compatible()` — cheap check before generating anything

Arity, host, trigger length window — nothing here folds. Checked per trigger set,
same as a real run would (a family is checked against every trigger set it might
build from).

In [ ]:
compatible_trigger_sets = []
incompatible = []
for ts in trigger_sets:
    compatibility = gate.is_compatible(ts, constraints)
    if compatibility.ok:
        compatible_trigger_sets.append(ts)
    else:
        incompatible.append((ts, compatibility))

print(f"{len(compatible_trigger_sets)}/{len(trigger_sets)} trigger set(s) compatible")
for ts, reason in incompatible[:5]:
    print(" incompatible:", ts.activators[0].trigger_id, reason)

assert compatible_trigger_sets, "no trigger set survived is_compatible"

## 5. `generate_designs()` — the full candidate pool

Called **once per compatible trigger set**, not just the top one — exactly how a real
run explores the search space, since a lower-ranked trigger can still fold into a
better switch. With `kozak_layouts=("trailing",)`, each trigger set yields one
`GateDesign` per `toehold_lengths` x `TRAILING_LOOP_LENGTHS` x `KOZAK_LINKER_LENGTHS`
combination it supports. Pooled together, this is the candidate set `engine.scoring`
would actually rank in a real run.

In [ ]:
designs = []
for ts in compatible_trigger_sets:
    designs.extend(gate.generate_designs(ts, constraints))

n_from_utr = sum(
    1 for d in designs if d.trigger_set.activators[0].start_index >= utr_start
)
print(f"{len(designs)} design(s) across {len(compatible_trigger_sets)} trigger(s)")
print(f"  {n_from_utr} design(s) trace back to a 3' UTR trigger")
print(f"  {len(designs) - n_from_utr} design(s) trace back to a CDS-region trigger")
print("\nfirst few:")
for d in designs[:5]:
    print(
        f"  {d.design_id:52} {d.length:3} nt  "
        f"trigger={d.trigger_set.activators[0].trigger_id:18}  "
        f"loop_len={d.architecture['loop_len']:2}  "
        f"kozak_linker_len={d.architecture['kozak_linker_len']}"
    )

## 6. `evaluate_design()` — raw metrics and sequences, across the whole pool

**Raw** values only — no normalising, weighting or ranking here, that is
`engine.scoring`'s job. Keys are exactly the metric names `DEFAULT_V1` declares.

For the `"trailing"` layout specifically, `predicted_leakage`/`dynamic_range` are read
from the **toehold+stem region**, not the AUG — the AUG sits outside the hairpin here
and stays roughly accessible whether or not the trigger is bound, so AUG-region
accessibility would not discriminate ON from OFF for this layout (see
`evaluate_design`'s docstring). This is a new, unreviewed proxy — not a port of
anything previously validated.

Because `PAYLOAD_CDS` is set (cell 4), every design below is folded **with the real
effector gene's own first `PAYLOAD_HEAD_LENGTH` nucleotides**, not a placeholder — the
molecule `gate_folding_energy`/`predicted_leakage`/`dynamic_range` are measured on is
the one a ribosome would actually encounter once this gene is attached. Set
`PAYLOAD_CDS = None` in cell 4 and re-run to see how the numbers shift for the exact
same designs without that information.

`ranked` below holds **every** design in the pool (hundreds, from 150 triggers), best
first by `dynamic_range` — the cell only *prints* the top `TOP_DISPLAY`, but nothing
downstream (the report, cell 7's follow-ups) is limited to that display slice.

In [ ]:
all_metrics = [(d, gate.evaluate_design(d)) for d in designs]
ranked = sorted(all_metrics, key=lambda pair: pair[1]["dynamic_range"], reverse=True)

TOP_DISPLAY = 20
print(f"top {TOP_DISPLAY} of {len(ranked)}:\n")
print(
    f"{'trigger':>18}  {'region':>6}  {'loop_len':>8}  {'linker_len':>10}  {'leakage':>8}  "
    f"{'dyn_range':>9}  {'folding_energy':>14}"
)
for d, m in ranked[:TOP_DISPLAY]:
    region = "utr" if d.trigger_set.activators[0].start_index >= utr_start else "cds"
    print(
        f"{d.trigger_set.activators[0].trigger_id:>18}  {region:>6}  "
        f"{d.architecture['loop_len']:>8}  {d.architecture['kozak_linker_len']:>10}  "
        f"{m['predicted_leakage']:>8.3f}  {m['dynamic_range']:>9.3f}  "
        f"{m['gate_folding_energy']:>14.1f}"
    )

print(f"\nsequences, same order, top {TOP_DISPLAY} (best first):\n")
for d, _m in ranked[:TOP_DISPLAY]:
    print(f"{d.design_id}  ({d.length} nt)")
    print(f"  {gate.emit_sequence(d)}")

# Not this gate's job to rank in a real run (engine.scoring owns that) — but picking the
# top-ranked one here to carry through the rest of this notebook's cells.
design, metrics = ranked[0]
print(f"\nbest by dynamic_range: {design.design_id}")
for name, value in metrics.items():
    print(f"  {name:22} {value}")

Three things worth noticing:

1. **A design finally clears `dynamic_range` 1.0.** The best of the 904,
   `eukaryotic_toehold-trig-AREG-563-30-12-trailing-8-0`, scores 1.548 with
   `predicted_leakage` of just 0.018 — a real, working switch by this proxy, not
   another below-1.0 result to explain away. Widening the search from 5 triggers to
   150 is what found it; it wasn't visible in the earlier, smaller pool.
2. **The best design came from the CDS region, not the 3&#8242; UTR**, and every one of
   the top 20 does too — the first 3&#8242; UTR-derived design lands at **rank 78** of
   904 (`dynamic_range` 0.109). The 41-candidate UTR floor (cell 7) guarantees 3&#8242;
   UTR triggers get a fair *chance* to compete; it doesn't guarantee they win, and for
   this transcript and mechanism they mostly don't. That's a real result about *AREG*
   specifically, not a flaw in the quota.
3. **Pool composition**: 904 designs total, 300 tracing back to a 3&#8242; UTR trigger
   and 604 to a CDS-region one (cell 13) — roughly proportional to the 41/109 trigger
   split, since every compatible trigger yields the same 4 designs (`loop_len` x
   `kozak_linker_len` for `toehold_length=12`; the 36 nt triggers also unlock
   `toehold_length` 15 in some cases, mattering little here).

## 7. `emit_sequence()` — the synthesis-ready sequence

In [ ]:
print(gate.emit_sequence(design))

The Kozak element and start codon aren't visually obvious in that raw string — for this
`"trailing"` layout they sit right after the closed hairpin, not inside it (contrast
with `"loop"`, where they'd be buried in the middle). Note also what comes right after
the AUG here: the real effector gene's own first `PAYLOAD_HEAD_LENGTH` nucleotides
(`eff` from `CERNAL_FUNCTIONS.py`, set as `PAYLOAD_CDS` in cell 4), not a placeholder
— `"trailing"` carries no trailing linker of its own (commit `ee2f5f6`); cap-dependent
scanning initiates the instant the 40S subunit meets Kozak+AUG, so nothing after the
start codon plays any role in finding it, and whatever comes after is either the
payload (when known, as folded in here) or nothing at all (commit `74bf7b4`). Locate
Kozak and the start codon explicitly using `design.architecture` (which records
`aug_index` exactly) and the gate's own `KOZAK_EUKARYOTIC` constant:

In [ ]:
seq = design.sequence
aug_index = design.architecture["aug_index"]
kozak_index = seq.find(gate.KOZAK_EUKARYOTIC)
assert kozak_index != -1, "Kozak element not found — architecture assumptions above are stale"

marks = [" "] * len(seq)
for i in range(kozak_index, kozak_index + len(gate.KOZAK_EUKARYOTIC)):
    marks[i] = "K"
for i in range(aug_index, aug_index + 3):
    marks[i] = "A"

print(seq)
print("".join(marks), " K = Kozak (GCCACC)   A = start codon (AUG)")

## 8. Report — top 15, with a 3&#8242; UTR floor and 2D structure

`ReportBuilder`/`StructureRenderer` (`src/engine/stages/reporting.py`, stage 6/S14)
are still stubs — `render_structure`, `render_circuit` and `build` all raise
`NotImplementedError("Step 5")`. This section is **not** that; it's a report built
here in the notebook from the real `ranked` pool computed above, not a pipeline
capability.

`ranked` is already best-first by `dynamic_range`. Taking the top 15 outright could
easily land all 15 in the CDS region (it very nearly does — see cell 18's finding
that the first 3&#8242; UTR design sits at rank 78/904), so the same reserved-floor
approach from trigger selection (cell 7) applies again here: keep the top 15,
but if fewer than `MIN_UTR_IN_REPORT` of them are 3&#8242; UTR-derived, swap in the
best-ranked 3&#8242; UTR designs not already present, evicting the *worst*-ranked
non-UTR entries to make room — never touching rank #1.

In [ ]:
def select_with_region_floor(ranked_pool, n, min_from_region, in_region):
    """Top `n` of `ranked_pool` (already best-first), with at least `min_from_region`
    satisfying `in_region`. If the natural top `n` doesn't have enough, swap in the
    best-ranked region matches not already present, evicting the worst-ranked
    non-matching entries — never the entries that were already in-region."""
    top = list(ranked_pool[:n])
    have = sum(1 for d, m in top if in_region(d))
    if have >= min_from_region:
        return top

    top_ids = {d.design_id for d, m in top}
    region_reserve = [
        pair for pair in ranked_pool if in_region(pair[0]) and pair[0].design_id not in top_ids
    ]
    non_region_in_top = [pair for pair in top if not in_region(pair[0])]

    for extra in region_reserve[: min_from_region - have]:
        if not non_region_in_top:
            break
        evict = non_region_in_top.pop()  # worst-ranked non-region entry (list end)
        top.remove(evict)
        top.append(extra)

    top.sort(key=lambda pair: pair[1]["dynamic_range"], reverse=True)
    return top


def in_3utr(design):
    return design.trigger_set.activators[0].start_index >= utr_start


MIN_UTR_IN_REPORT = 3
REPORT_SIZE = 15
report = select_with_region_floor(ranked, REPORT_SIZE, MIN_UTR_IN_REPORT, in_3utr)

n_utr_in_report = sum(1 for d, m in report if in_3utr(d))
print(f"report: {len(report)} design(s), {n_utr_in_report} from the 3' UTR "
      f"(floor was {MIN_UTR_IN_REPORT})\n")

print(
    f"{'#':>3}  {'trigger':>18}  {'region':>6}  {'geometry (nt)':>14}  "
    f"{'leakage':>8}  {'dyn_range':>9}  {'ΔG (kcal/mol)':>16}  {'gc%':>6}"
)
for rank, (d, m) in enumerate(report, 1):
    region = "utr" if in_3utr(d) else "cds"
    arch = d.architecture
    geometry = f"{arch['toehold_length']}/{arch['loop_len']}/{arch['kozak_linker_len']}"
    print(
        f"{rank:>3}  {d.trigger_set.activators[0].trigger_id:>18}  {region:>6}  "
        f"{geometry:>14}  {m['predicted_leakage']:>8.3f}  {m['dynamic_range']:>9.3f}  "
        f"{m['gate_folding_energy']:>16.1f}  {m['gc_content']:>6.1f}"
    )

### 2D structure

`RNA.get_xy_coordinates` (ViennaRNA's own layout algorithm, not a house rule
violation — this is display-only, not a scientific computation, and this notebook
isn't `src/engine/`, so the "only two modules import RNA" rule (`CLAUDE.md` §5)
doesn't apply here) turns a dot-bracket string into 2D coordinates. Each base is
coloured by identity; Kozak and the start codon are outlined so the mechanism this
whole layout exists for is visible at a glance.

**This plots `folder.mfe(design.sequence).structure` — the single lowest-energy
predicted fold — not `design.dot_bracket`.** The two are very different:
`dot_bracket` is the generator's *intended* target structure (the same idealized
toehold-hairpin shape for every design of matching geometry, regardless of how well
it actually folds); the MFE structure is a real prediction, and for this
construction it folds far more extensively than intended — the leader and toehold
pair up with downstream sequence well beyond the designed stem. It's still not
identical to what `evaluate_design` measures — that reads ensemble base-pair
*probabilities* from the partition function, not one single lowest-energy structure
— but it's a real, representative fold, not a fabricated one, and the single most
informative structure ViennaRNA can hand back for a static picture. Plotting the
target instead would have shown all 15 designs as visually near-identical, hiding
exactly the differences that explain their metrics.

In [ ]:
import RNA
import matplotlib.pyplot as plt

BASE_COLORS = {"A": "#FFD6A5", "U": "#A0C4FF", "G": "#FFADAD", "C": "#CAFFBF"}


def plot_structure_mini(ax, design, label):
    seq = design.sequence
    structure = gate.folder.mfe(seq).structure  # real predicted fold, not the target
    coords = RNA.get_xy_coordinates(structure)
    xs = [coords.get(i).X for i in range(len(structure))]
    ys = [coords.get(i).Y for i in range(len(structure))]

    ax.plot(xs, ys, "-", color="#B9C2BC", linewidth=1.1, zorder=1)

    stack = []
    for i, c in enumerate(structure):
        if c == "(":
            stack.append(i)
        elif c == ")":
            j = stack.pop()
            ax.plot(
                [xs[i], xs[j]], [ys[i], ys[j]], "-", color="#8899AA",
                linewidth=0.9, zorder=1, alpha=0.7,
            )

    aug_index = design.architecture["aug_index"]
    kozak_index = seq.find(gate.KOZAK_EUKARYOTIC)
    highlight = set(range(kozak_index, kozak_index + len(gate.KOZAK_EUKARYOTIC)))
    highlight |= set(range(aug_index, aug_index + 3))

    for i, base in enumerate(seq):
        is_hl = i in highlight
        ax.scatter(
            xs[i], ys[i],
            s=26 if is_hl else 14,
            color=BASE_COLORS.get(base, "#EEEEEE"),
            edgecolors="#1A2620" if is_hl else "#666666",
            linewidths=1.1 if is_hl else 0.3,
            zorder=3 if is_hl else 2,
        )

    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(label, fontsize=9, fontweight="bold", loc="left")


fig, axes = plt.subplots(3, 5, figsize=(18, 11))
for i, (d, _m) in enumerate(report):
    region = "UTR" if in_3utr(d) else "CDS"
    label = f"#{i + 1}  {d.trigger_set.activators[0].trigger_id}  ({region})"
    plot_structure_mini(axes[i // 5][i % 5], d, label)

fig.suptitle(
    "Top 15 trailing-Kozak toeholds against AREG — real MFE fold, colour = base "
    "identity, outlined = Kozak/AUG",
    fontsize=12,
)
plt.tight_layout()
plt.show()

Base colours: A `#FFD6A5` · U `#A0C4FF` · G `#FFADAD` · C `#CAFFBF` (matching the
convention already used in `CERNAL_FUNCTIONS.py`'s own `plot_rna_structure`).

The correlation with the metrics table is visible directly in these diagrams: #1
(`dynamic_range` 1.55) folds into one clean extended hairpin with Kozak/AUG exposed
in a small accessible loop of their own. The three 3&#8242; UTR designs pulled in to meet
`MIN_UTR_IN_REPORT` land at #13&#8211;#15 — the worst `dynamic_range` in this 15 — and
their folds show why: Kozak/AUG buried inside a branched, multi-junction structure
rather than sitting in a clean loop, visibly less reachable, which is exactly what a
low `dynamic_range` means for this proxy. This is the real predicted fold explaining
the ranking, not the ranking asserted and then illustrated after the fact.

## Where this fits in a real run

This notebook now pools 904 designs across 150 triggers (41 from the 3&#8242; UTR, 109
from the CDS region) and one layout — closer to a real run, but still a slice of it.
In the actual pipeline:

- Every trigger `TriggerScorer` keeps (not just the top `N_TRIGGERS`) gets checked
  with `is_compatible` and, if it passes, run through `generate_designs` — and by
  default that sweeps **both** `"loop"` and `"trailing"` layouts (this notebook
  restricted to `"trailing"` only via `kozak_layouts` — see cell 4). No 3&#8242; UTR
  quota exists anywhere in `engine.stages.triggers` — the floor enforced here (cell 7)
  is a notebook-level policy, not an engine feature; if a real run needs this
  guarantee, it belongs in `Constraints` or `TriggerScorer` itself, not reimplemented
  per caller.
- Every design's raw metrics go through `engine.scoring` — `build_metrics`,
  `weighted_score`, `failed_filter`, `rank_candidates` — which is what actually
  decides which designs survive and how they rank, comparably with every other gate
  family's designs, every trigger, **and across layouts**. Neither this notebook nor
  the gate itself picks a winner — sorting by `dynamic_range` alone (cell 17) is a
  stand-in for that, not the real ranking.
- `CandidateStore` records provenance and writes the stage snapshot; nothing here
  hand-rolls a results CSV.
- The **payload** folds into evaluation when known (`PAYLOAD_CDS` in cell 4) but the
  *actual* fused construct is still assembled later, at plasmid assembly
  (`PlasmidBuilder.build(circuit, DesiredOutcome.CUSTOM, custom_payload=...)`), using
  the complete gene, not just its folded-in head.

The two-input AND version (`EukaryoticToeholdAndGate`) is **not** implemented yet —
its `generate_designs` still raises `NotImplementedError("Step 5")`. This notebook
only covers the single-input case.

Open questions this work surfaced, not resolved here (see commits `d754812`,
`ee2f5f6`, `74bf7b4`): whether `"loop"` should still ship as the eukaryotic default
now that `"trailing"` exists (and now that `"loop"`'s Kozak-AUG adjacency is known to
be broken); whether `TRAILING_LOOP_LENGTHS`/`KOZAK_LINKER_LENGTHS` are the right
ranges to sweep; whether the `"trailing"` leakage proxy (toehold+stem accessibility)
is the right measurement for scanning-ribosome blockage at all; whether fusing the
payload directly after the switch's own placeholder AUG (rather than dropping that AUG
in favour of the payload's own) is the right call; the CDS boundary here is a
longest-ORF heuristic, not real annotation — wrong for any transcript with a shorter
true CDS or a non-canonical start; and whether a 3&#8242; UTR floor belongs in
`Constraints` as a first-class, engine-level concept rather than a per-notebook
policy, given the trigger scoring map's own emphasis on where a trigger sits along the
transcript.